**Przetwarzanie strumieni danych lista 7 Oliwia Borkowska**

**Zad 1** Przygotuj kod w Pythonie, który wyświetli następujące typy falek: Haar, Daubechies, Symlets, Coiflets, Biortogonalna, Gaussian, Meksykański kapelusz, Morleta. W celu rozwiązania zadania można wykorzystać pakiet pywt

In [64]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
import scipy.signal as signal
import ipywidgets as widgets
from ipywidgets import interact

types_with_order = ['db', 'sym', 'coif', 'gaus']

wavelet_versions = {
    'haar': ['haar'],
    'db': [f'db{i}' for i in range(1, 11)],
    'sym': [f'sym{i}' for i in range(2, 9)],
    'coif': [f'coif{i}' for i in range(1, 6)],
    'bior': ['bior1.3','bior1.5','bior2.2', 'bior2.4', 'bior2.6', 'bior3.1', 'bior3.3','bior3.5', 'bior3.7','bior4.4'],
    'gaus': [f'gaus{i}' for i in range(1, 4)],
    'mexh': ['mexh'],
    'morl': ['morl']
}

def rysowanie_falek(wavelet_type='haar', wavelet_version='haar', order=1):
    if wavelet_type in types_with_order:
        if wavelet_type == 'gaus':
            wavelet_version = f'gaus{order}'
        elif wavelet_type == 'sym':
            wavelet_version = f'sym{order}'
        elif wavelet_type == 'coif':
            wavelet_version = f'coif{order}'
        else:
            wavelet_version = f'db{order}'

    plt.figure(figsize=(14, 7))
    plt.suptitle(f'Falka: {wavelet_version} - Typ: {wavelet_type}', fontsize=16)

    if wavelet_version.startswith('gaus'):
        t = np.linspace(-5, 5, 1000)
        std = int(wavelet_version[-1])
        psi = signal.gaussian(1000, std=std)
        plt.plot(t, psi, label=f'Falka Gaussa (rząd {std})')

    elif wavelet_version == 'mexh':
        t = np.linspace(-5, 5, 1000)
        psi = (1 - t**2) * np.exp(-t**2 / 2)
        plt.plot(t, psi, label='Falka Mexican Hat')

    elif wavelet_version == 'morl':
        t = np.linspace(-5, 5, 1000)
        psi = signal.morlet(len(t), w=5)
        plt.plot(t, psi, label='Falka Morlet')

    else:
        wavelet_obj = pywt.Wavelet(wavelet_version)
        wavefun_output = wavelet_obj.wavefun(level=5)

        if len(wavefun_output) == 3:
            phi, psi, x = wavefun_output
            plt.plot(x, psi, label='Funkcja falkowa ψ', color='magenta')
            plt.plot(x, phi, label='Funkcja skalująca φ', linestyle='--', color='skyblue', alpha=0.7)
        elif len(wavefun_output) == 2:
            psi, x = wavefun_output
            plt.plot(x, psi, label='Falka')
        else:
            psi, x = wavefun_output[:2]
            plt.plot(x, psi, label='Falka')

    plt.legend()
    plt.grid(True)
    plt.show()

def falki_interaktywnosc():
    wavelet_type_widget = widgets.Dropdown(
        options=list(wavelet_versions.keys()),
        value='haar',
        description='Typ falki:'
    )

    wavelet_version_widget = widgets.Dropdown(
        options=wavelet_versions['haar'],
        value='haar',
        description='Wersja:'
    )

    order_slider = widgets.IntSlider(
        value=2,
        min=1,
        max=10,
        step=1,
        description='Rząd:'
    )
    
# suwak
    def update_type(change):
        typ = change['new']
        wavelet_version_widget.options = wavelet_versions[typ]
        wavelet_version_widget.value = wavelet_versions[typ][0]
        if typ in types_with_order:
            order_slider.layout.display = 'block'
            if typ == 'sym':
                order_slider.min, order_slider.max = 2, 8
            elif typ == 'coif':
                order_slider.min, order_slider.max = 1, 5
            elif typ == 'gaus':
                order_slider.min, order_slider.max = 1, 3
            else:  # db
                order_slider.min, order_slider.max = 1, 10
        else:
            order_slider.layout.display = 'none'

    wavelet_type_widget.observe(update_type, names='value')

    interact(
        rysowanie_falek,
        wavelet_type=wavelet_type_widget,
        wavelet_version=wavelet_version_widget,
        order=order_slider
    )

falki_interaktywnosc()

interactive(children=(Dropdown(description='Typ falki:', options=('haar', 'db', 'sym', 'coif', 'bior', 'gaus',…

**Zad. 2** Przygotuj kod w Pythonie, który wyświetli falkę Daubechies w różnych wersjach (db1, db2, itd.) i dla różnych parametrów.

In [28]:
import pywt
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Checkbox

def rysowanie_falki_daubechies(rzad, level, pokaz_wavelet, pokaz_scaling, pokaz_widmo):
    try:
        nazwa = 'db' + str(rzad)
        falka = pywt.Wavelet(nazwa)

        phi, psi, x = falka.wavefun(level=level)

        fig, axs = plt.subplots(1, 2 if pokaz_widmo else 1, figsize=(14 if pokaz_widmo else 8, 4))

        ax = axs[0] if pokaz_widmo else axs
        if pokaz_wavelet:
            ax.plot(x, psi, label="ψ (falka)", color='magenta')
        if pokaz_scaling:
            ax.plot(x, phi, label="φ (funkcja skalująca)", color='skyblue', alpha=0.7)

        ax.set_title(f"Falka Daubechies: {nazwa} (poziom={level})")
        ax.grid(True)
        ax.legend()

        if pokaz_widmo:
            N = len(psi)
            fft = np.fft.fft(psi)
            freq = np.fft.fftfreq(N, d=(x[1] - x[0]))
            axs[1].plot(np.fft.fftshift(freq), np.fft.fftshift(np.abs(fft)), color='blue')
            axs[1].set_title("Widmo amplitudowe ψ")
            axs[1].set_xlabel("Częstotliwość")
            axs[1].set_ylabel("Amplituda")
            axs[1].grid(True)

        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"Błąd: {e}")

# suwak
interact(
    rysowanie_falki_daubechies,
    rzad=IntSlider(value=1, min=1, max=20, step=1, description="Rząd (dbN):"),
    level=IntSlider(value=5, min=1, max=10, step=1, description="Poziom:"),
    pokaz_wavelet=Checkbox(value=True, description="Pokaż ψ"),
    pokaz_scaling=Checkbox(value=True, description="Pokaż φ"),
    pokaz_widmo=Checkbox(value=True, description="Pokaż widmo")
)

interactive(children=(IntSlider(value=1, description='Rząd (dbN):', max=20, min=1), IntSlider(value=5, descrip…

<function __main__.rysuj_falke_daubechies(rzad, level, pokaz_wavelet, pokaz_scaling, pokaz_widmo)>

**Zad 3.** Przygotuj kod w Pythonie, który dokona dekompozycji sygnału świergotliwego (chirp signal) z wykorzystaniem trzech różnych falek. Uzyskane wyniki wyświetl w czytelnej postaci.

In [39]:
import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.signal import chirp
from ipywidgets import interact, IntSlider

def dekompozycja_chirp(f1):
    t = np.linspace(0, 1, 1024)
    sygnal = chirp(t, f0=6, f1=f1, t1=1, method='linear')

    falka_lista = ['haar', 'db4', 'sym5', 'coif3', 'bior3.3']

    fig, axs = plt.subplots(len(falka_lista), 3, figsize=(15, 3 * len(falka_lista)))
    fig.suptitle(f"Dekompozycja chirp (f0=6 Hz, f1={f1} Hz) dla różnych falek", fontsize=16)

    for i, falka_nazwa in enumerate(falka_lista):
        cA, cD = pywt.dwt(sygnal, falka_nazwa)
        t_cA = np.linspace(0, 1, len(cA))
        t_cD = np.linspace(0, 1, len(cD))

        axs[i, 0].plot(t, sygnal, color='purple')
        axs[i, 0].set_title(f"Oryginalny sygnał – {falka_nazwa}")

        axs[i, 1].plot(t_cA, cA, color='magenta')
        axs[i, 1].set_title("Aproksymacja (cA)")

        axs[i, 2].plot(t_cD, cD, color='pink')
        axs[i, 2].set_title("Detale (cD)")

        for ax in axs[i]:
            ax.grid(True)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# suwak
interact(dekompozycja_chirp, f1=IntSlider(min=10, max=200, step=5, value=80, description='f1 (Hz)'))


interactive(children=(IntSlider(value=80, description='f1 (Hz)', max=200, min=10, step=5), Output()), _dom_cla…

<function __main__.dekompozycja_chirp(f1)>